In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

sc.settings.verbosity = 0

In [2]:
import GenKI as gk
from GenKI.preprocesing import build_adata
from GenKI.dataLoader import DataLoader
from GenKI.train import VGAE_trainer
from GenKI import utils

%load_ext autoreload
%autoreload 2

/home/siamsoha/miniconda3/envs/ogenki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/siamsoha/miniconda3/envs/ogenki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/siamsoha/miniconda3/envs/ogenki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  return module_get_attr_redirect(attr_name, deprecated_mapping=_DEPRECATED)
/home/siamsoha/miniconda3/envs/ogenki/lib/python3.12/site-packages/anndata/__init__.py:70: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.

In [ ]:
# # example

# data_dir = os.path.join("data", "filtered_gene_bc_matrices")
# counts_path = str(os.path.join(data_dir, "X.txt"))
# gene_path = str(os.path.join(data_dir, "g.txt"))
# cell_path = str(os.path.join(data_dir, "c.txt"))

# adata = build_adata(counts_path, 
#                     gene_path, 
#                     cell_path, 
#                     meta_cell_cols=["cell_type"], # colname of cell type
#                     delimiter=',', # X.txt
#                     transpose=True, # X.txt
#                     log_normalize=True,
#                     scale_data=True,
#                    )

# adata = adata[:100, :300].copy()
# print(adata)

load counts from data\filtered_gene_bc_matrices\X.txt
AnnData object with n_obs × n_vars = 100 × 300
    obs: 'cell_type'
    uns: 'log1p'
    layers: 'raw', 'norm'


In [43]:
adata = sc.read_h5ad("/home/siamsoha/Siam/Genki/GenKI-master/adata_coronary_symbols.h5ad")
adata

AnnData object with n_obs × n_vars = 35920 × 25840
    obs: 'donor_id', 'dataset', 'cell_type_level1', 'cell_type_level2', 'origin', 'sorting', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'suspension_type', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'size_factors', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'original_gene_names', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'feature_id_original'
    uns: 'cell_type_level1_colors', 'citation', 'dataset_colors', 'organism', 'organism_ontology_term_id', 'origin_colors', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_scpoli'

In [44]:
adata.obs['dataset'].unique()

['Wirka', 'Emoto', 'Chowdhury']
Categories (3, object): ['Chowdhury', 'Emoto', 'Wirka']

In [45]:
# Count cells per dataset
dataset_counts = adata.obs['dataset'].value_counts()
print(dataset_counts)

dataset
Chowdhury    22548
Wirka        11751
Emoto         1621
Name: count, dtype: int64


In [ ]:
# # Subset Chowdhury dataset into new cdata
# cdata = adata[adata.obs['dataset'] == 'Chowdhury'].copy()
# print(f"Chowdhury dataset subset: {cdata.n_obs} cells × {cdata.n_vars} genes")
# cdata

Chowdhury dataset subset: 22548 cells × 25840 genes


AnnData object with n_obs × n_vars = 22548 × 25840
    obs: 'donor_id', 'dataset', 'cell_type_level1', 'cell_type_level2', 'origin', 'sorting', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'suspension_type', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'size_factors', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'original_gene_names', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'feature_id_original'
    uns: 'cell_type_level1_colors', 'citation', 'dataset_colors', 'organism', 'organism_ontology_term_id', 'origin_colors', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_scpoli'

In [46]:
# Subset Emoto dataset into new edata
edata = adata[adata.obs['dataset'] == 'Emoto'].copy()
print(f"Emoto dataset subset: {edata.n_obs} cells × {edata.n_vars} genes")
edata

Emoto dataset subset: 1621 cells × 25840 genes


AnnData object with n_obs × n_vars = 1621 × 25840
    obs: 'donor_id', 'dataset', 'cell_type_level1', 'cell_type_level2', 'origin', 'sorting', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'suspension_type', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'size_factors', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'original_gene_names', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'feature_id_original'
    uns: 'cell_type_level1_colors', 'citation', 'dataset_colors', 'organism', 'organism_ontology_term_id', 'origin_colors', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_scpoli',

In [47]:
adata = edata.copy()

In [48]:
# Select top 1000 highly variable genes
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
adata = adata[:, adata.var['highly_variable']].copy()

print(f"After HVG selection: {adata.n_obs} cells × {adata.n_vars} genes")
adata

After HVG selection: 1621 cells × 2000 genes


/tmp/ipykernel_5229/2686291186.py:2: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')


AnnData object with n_obs × n_vars = 1621 × 2000
    obs: 'donor_id', 'dataset', 'cell_type_level1', 'cell_type_level2', 'origin', 'sorting', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'suspension_type', 'assay_ontology_term_id', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'cell_type_ontology_term_id', 'is_primary_data', 'size_factors', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'n_cells', 'original_gene_names', 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'feature_id_original', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'cell_type_level1_colors', 'citation', 'dataset_colors', 'organism', 'organism_ontology_term_id', 'o

In [49]:
from GenKI.preprocesing import build_adata

# Preprocess your raw adata
adata = build_adata(
    adata,                    # Your raw AnnData object
    log_normalize=True,       # Optional: log-normalize counts
    scale_data=True,          # Required: creates adata.layers["norm"] and standardizes adata.X
    as_sparse=True,           # Optional: store as sparse matrix
    uppercase=True,           # Optional: convert gene names to uppercase
)

In [50]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                953
Monocyte              218
Macrophage            155
B cell                146
NK cell                88
Dendritic cell         36
Plasma cell            14
Neutrophil              6
Mast cell               3
Smooth Muscle Cell      2
Name: count, dtype: int64

In [51]:
# Search for GZMK in var names
gzmk_indices = np.where(adata.var_names.str.contains('GZMK'))[0]
print(f"GZMK found at indices: {gzmk_indices}")

# If found, display the gene names
if len(gzmk_indices) > 0:
    print("GZMK gene names:")
    for idx in gzmk_indices:
        print(f"Index {idx}: {adata.var_names[idx]}")
else:
    print("GZMK not found in var names")

GZMK found at indices: [304]
GZMK gene names:
Index 304: GZMK


In [52]:
# gene to ko
adata.var_names[304]

'GZMK'

In [53]:
# load data

data_wrapper =  DataLoader(
                adata, # adata object
                target_gene = [304], # KO gene name/index
                target_cell = None, # obsname for cell type, if none use all
                obs_label = "cell_type", # colname for genes
                GRN_file_dir = "GRNs", # folder name for GRNs
                rebuild_GRN = True, # whether build GRN by pcNet
                pcNet_name = "pcNet_example", # GRN file name
                verbose = True, # whether verbose
                n_cpus = 12, # multiprocessing
                )

data_wt = data_wrapper.load_data()
data_ko = data_wrapper.load_kodata()

use all the cells (1621) in adata
build GRN


2026-02-20 03:43:20,528	INFO worker.py:2013 -- Started a local Ray instance.
/home/siamsoha/miniconda3/envs/ogenki/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


ray init, using 12 CPUs
execution time of making pcNet: 1524.87 s
GRN has been built and saved in "GRNs/pcNet_example.npz"
init completed



In [54]:
# init trainer

hyperparams = {"epochs": 100, 
               "lr": 7e-4, 
               "beta": 1e-4, 
               "seed": 8096}
log_dir=None 

sensei = VGAE_trainer(data_wt, 
                     epochs=hyperparams["epochs"], 
                     lr=hyperparams["lr"], 
                     log_dir=log_dir, 
                     beta=hyperparams["beta"],
                     seed=hyperparams["seed"],
                     verbose=True,
                     )

In [55]:
# %%timeit
sensei.train()

Epoch: 000, Loss: 1.9034, AUROC: 0.7652, AP: 0.6202
Epoch: 001, Loss: 1.8069, AUROC: 0.8047, AP: 0.6628
Epoch: 002, Loss: 1.7381, AUROC: 0.8415, AP: 0.7101
Epoch: 003, Loss: 1.6422, AUROC: 0.8730, AP: 0.7589
Epoch: 004, Loss: 1.6206, AUROC: 0.8947, AP: 0.7960
Epoch: 005, Loss: 1.6265, AUROC: 0.9063, AP: 0.8175
Epoch: 006, Loss: 1.5053, AUROC: 0.9127, AP: 0.8298
Epoch: 007, Loss: 1.4893, AUROC: 0.9154, AP: 0.8345
Epoch: 008, Loss: 1.4445, AUROC: 0.9160, AP: 0.8349
Epoch: 009, Loss: 1.4198, AUROC: 0.9158, AP: 0.8334
Epoch: 010, Loss: 1.3962, AUROC: 0.9158, AP: 0.8324
Epoch: 011, Loss: 1.4139, AUROC: 0.9168, AP: 0.8335
Epoch: 012, Loss: 1.3638, AUROC: 0.9198, AP: 0.8391
Epoch: 013, Loss: 1.3608, AUROC: 0.9250, AP: 0.8497
Epoch: 014, Loss: 1.3391, AUROC: 0.9314, AP: 0.8634
Epoch: 015, Loss: 1.2928, AUROC: 0.9383, AP: 0.8791
Epoch: 016, Loss: 1.2772, AUROC: 0.9448, AP: 0.8955
Epoch: 017, Loss: 1.2653, AUROC: 0.9505, AP: 0.9095
Epoch: 018, Loss: 1.2616, AUROC: 0.9547, AP: 0.9201
Epoch: 019, 

In [56]:
sensei.save_model('model_emoto')

save model parameters to model/model_emoto.th


In [58]:
# get distance between wt and ko

z_mu_wt, z_std_wt = sensei.get_latent_vars(data_wt)
z_mu_ko, z_std_ko = sensei.get_latent_vars(data_ko)
dis = gk.utils.get_distance(z_mu_ko, z_std_ko, z_mu_wt, z_std_wt, by="KL")

In [59]:
# raw ranked gene list

res_raw = utils.get_generank(data_wt, dis, rank=True)
res_raw.head()

,dis,rank
GZMK,4621.349542,1
VCAN,0.892030,2
HLA-DQA1,0.887954,3
HLA-DQA2,0.830548,4
FCN1,0.823312,5


In [60]:
# if permutation test

null = sensei.pmt(data_ko, n=100, by="KL")
res = utils.get_generank(data_wt, dis, null,)
#                       save_significant_as = 'gene_list_example')
res

Permutating:   0%|          | 0/100 [00:00<?, ?it/s]

Permutating: 100%|██████████| 100/100 [00:04<00:00, 23.78it/s]


,dis,index,hit,rank
GZMK,4621.349542,304,100,1
HLA-DPB1,0.800120,1401,95,2
HLA-DPA1,0.786744,1456,95,3
SELL,0.665496,1181,97,4
NKG7,0.265831,225,95,5


In [61]:
res = utils.get_generank(data_wt, dis, null, cutoff=0)  # Include all genes with hits

In [66]:
res.to_csv("res.csv", index=True)

In [65]:
res

,dis,index,hit,rank
GZMK,4621.349542,304,100,1
VCAN,0.892030,51,84,2
HLA-DQA1,0.887954,1210,85,3
HLA-DQA2,0.830548,1508,94,4
FCN1,0.823312,126,85,5
...,...,...,...,...
BIRC5,0.000004,137,1,459
PTPRF,0.000004,618,1,460
SYNM,0.000004,1094,1,461
CCDC136,0.000004,472,1,462
